In [1]:
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

In [2]:
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

url = "https://www.cars24.com/buy-used-cars-new-delhi/"
driver.get(url)

time.sleep(5)
print("Browser launched and page loaded!")

Browser launched and page loaded!


In [3]:
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

print(soup.prettify()[:500])

<html lang="en">
 <head>
  <script async="" src="https://connect.facebook.net/signals/config/1594462310820300?v=2.9.373&amp;r=stable&amp;domain=www.cars24.com&amp;hme=6d1ed5deee7eafc01c53d9e9fc3c4e10db50c3d28237bc4932686777e5c3969a&amp;ex_m=111%2C215%2C163%2C23%2C76%2C77%2C154%2C72%2C71%2C11%2C172%2C96%2C17%2C146%2C134%2C41%2C79%2C84%2C142%2C168%2C174%2C27%2C15%2C28%2C29%2C30%2C32%2C50%2C155%2C81%2C119%2C19%2C21%2C46%2C42%2C44%2C43%2C89%2C98%2C102%2C117%2C153%2C156%2C48%2C118%2C25%2C22%2C126%2C7


In [4]:
car_cards = soup.find_all('div', class_="styles_contentWrap__9oSrl")

print(f"Found {len(car_cards)} cars on this page.")
if len(car_cards) > 0:
    first_car = car_cards[0]
    print("Successfully isolated the first car!")
else:
    print("No cars found. Double check your class name.")

Found 40 cars on this page.
Successfully isolated the first car!


In [5]:
car_data = {}

try :
    title_tag = first_car.find('span', class_='sc-ksBlki dxpZAa') 
    car_data['Title']=title_tag.text.strip() if title_tag else None

    price_tag = first_car.find('div',class_='styles_priceWrap__VwWBV')
    car_data['Price'] = price_tag.text.strip() if price_tag else None

    specs_list = first_car.find('ul', class_='sc-gsGlKM hCHamb')

    if specs_list:
        specs = specs_list.find_all('p')
        car_data['Kilometers'] = specs[0].text.strip() if len(specs) > 0 else None
        car_data['Fuel_Type'] = specs[1].text.strip() if len(specs) > 1 else None
        car_data['Transmission'] = specs[2].text.strip() if len(specs) > 2 else None

    print("Extraction Successful")
    print(car_data)
except Exception as e:
    print(f"Error during extraction: {e}")

Extraction Successful
{'Title': '2015 Hyundai Eon', 'Price': '₹1.88L₹1.72 lakh'}


In [8]:
lst =[]

for i in range(len(car_cards)):
    data_car={}
    car_detail = car_cards[i]

    title = car_detail.find('span',class_="sc-ksBlki dxpZAa")
    data_car['Title'] = title.text.strip() if title else None

    price = car_detail.find('div',class_="styles_priceWrap__VwWBV")
    data_car["Price"] = price.text.strip() if price else None
    spec_list = car_detail.find('ul',class_="sc-gJqSRp wIveL")

    if spec_list:
        specs = spec_list.find_all('p')
        data_car['Kilometers'] = specs[0].text.strip() if len(specs) > 0 else None
        data_car['Fuel_Type'] = specs[1].text.strip() if len(specs) > 1 else None
        data_car['Transmission'] = specs[2].text.strip() if len(specs) > 2 else None

    lst.append(data_car)
pd.DataFrame(lst)

,Title,Price,Kilometers,Fuel_Type,Transmission
0,2015 Hyundai Eon,₹1.88L₹1.72 lakh,"43,518 km",Petrol,Manual
1,2017 Maruti Swift,₹2.70L₹2.23 lakh,"79,321 km",Petrol,Manual
2,2021 Tata Tiago,₹3.75L₹3.04 lakh,"70,488 km",Petrol,Manual
3,2018 Datsun Redi Go,₹1.85L₹1.64 lakh,"60,214 km",CNG,Manual
4,2020 Renault TRIBER,₹3.56L₹2.91 lakh,"39,097 km",Petrol,Manual
5,2015 Maruti Wagon R 1.0,₹1.95L₹1.53 lakh,"84,560 km",Petrol,Manual
6,2014 Hyundai Grand i10,₹2.05L₹1.61 lakh,"58,791 km",Petrol,Manual
7,2022 Nissan MAGNITE,₹3.85L₹3.58 lakh,"54,847 km",Petrol,Manual
8,2016 Ford Ecosport,₹2.98L₹2.49 lakh,"1,04,834 km",Petrol,Manual
9,2018 Tata Tiago,₹3.05L₹2.94 lakh,"60,058 km",Petrol,Manual


In [10]:
df= pd.DataFrame(lst)
df.to_csv(r'C:\Programming\Machine Learning\Used Car Price Intelligence\raw_car_listing.csv', index= False)
print("Raw data saved Successfully")

Raw data saved Successfully
